In [1]:
import xgboost as xgb
from statsmodels.tools.eval_measures import rmse
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv("../data/FRB_H15.csv").dropna()

#rename columns
df.rename(columns={"Series Description": "Date", "Market yield on U.S. Treasury securities at 1-month  constant maturity, quoted on investment basis": "0Y1M", "Market yield on U.S. Treasury securities at 3-month  constant maturity, quoted on investment basis": "0Y3M", "Market yield on U.S. Treasury securities at 6-month  constant maturity, quoted on investment basis": "0Y6M", "Market yield on U.S. Treasury securities at 1-year  constant maturity, quoted on investment basis": "1Y", "Market yield on U.S. Treasury securities at 2-year  constant maturity, quoted on investment basis": "2Y", "Market yield on U.S. Treasury securities at 3-year  constant maturity, quoted on investment basis": "3Y", "Market yield on U.S. Treasury securities at 5-year  constant maturity, quoted on investment basis": "5Y", "Market yield on U.S. Treasury securities at 7-year  constant maturity, quoted on investment basis": "7Y", "Market yield on U.S. Treasury securities at 10-year  constant maturity, quoted on investment basis": "10Y", "Market yield on U.S. Treasury securities at 20-year constant maturity, quoted on investment basis": "20Y", "Market yield on U.S. Treasury securities at 30-year  constant maturity, quoted on investment basis": "30Y"}, inplace=True)
df.rename(columns={"Market yield on U.S. Treasury securities at 20-year  constant maturity, quoted on investment basis": "20Y"}, inplace=True)

#make index to datetime for timeseries
df["Date"] = pd.to_datetime(df["Date"])
df.set_index("Date", inplace=True)

df.tail(10)

FileNotFoundError: [Errno 2] No such file or directory: '../data/FRB_H15.csv'

In [ ]:
"""
L(t) = 1
S(t) = 1-e^(-xt)/xt
C(t) = S(t) - e^(-xt) 

x = 0.0609
"""

maturities = [1/12, 3/12, 6/12, 1, 2, 3, 5, 7, 10, 20, 30]
x = 0.0609
A = np.array([
    [
    1,
    (1 - np.exp(-x * t)) / (x * t),
    ((1 - np.exp(-x * t)) / (x * t)) - np.exp(-x * t),
    ]
    for t in maturities
])


rows = []

for row in df.itertuples():
    # get actual yields for each date
    y = [row._1, row._2, row._3, row._4, row._5, row._6, row._7, row._8, row._9, row._10, row._11]

    betas, *_ = np.linalg.lstsq(A, y, rcond=None)
    b1, b2, b3 = betas
    rows.append({'Date': row.Index, 'Beta 1': b1, 'Beta 2': b2, 'Beta 3': b3})

betas = pd.DataFrame(rows).set_index("Date")
# betas.shift(-5)
betas.tail()



,Beta 1,Beta 2,Beta 3
Date,,,
2026-05-08,5.270308,-1.599870,1.604442
2026-05-11,5.058645,-1.366787,2.039306
2026-05-12,4.857632,-1.157728,2.533876
2026-05-13,4.870180,-1.178757,2.539232
2026-05-14,4.700093,-1.001930,2.771229


In [ ]:
# 1 Day horizon for Beta 1
x = []
y = []
for i in range(0, len(betas) - 1):
    x.append(betas.iloc[i].values[0]) 
    y.append(betas.iloc[i+1].values[0])

# print(x)
# print(y)

x = np.array(x)
y = np.array(y)
x_arr = np.column_stack((np.ones(x.shape[0]), x))

X_train, X_test, y_train, y_test = train_test_split(x_arr, y, test_size = 20, shuffle=False)


model = xgb.XGBRegressor()
model.fit(X_train, y_train)

predictions = model.predict(X_test)

beta_1_rmse = rmse(y_test, predictions)



In [ ]:
# 1 Day horizon for Beta 2 
x = []
y = []
for i in range(0, len(betas) - 1):
    x.append(betas.iloc[i].values[1]) 
    y.append(betas.iloc[i+1].values[1])

# print(x)
# print(y)

x = np.array(x)
y = np.array(y)
x_arr = np.column_stack((np.ones(x.shape[0]), x))

X_train, X_test, y_train, y_test = train_test_split(x_arr, y, shuffle=False)


model = xgb.XGBRegressor()
model.fit(X_train, y_train)

predictions = model.predict(X_test)

beta_2_rmse = rmse(y_test, predictions)



In [ ]:
# 1 Day horizon for Beta 3 
x = []
y = []
for i in range(0, len(betas) - 1):
    x.append(betas.iloc[i].values[2]) 
    y.append(betas.iloc[i+1].values[2])

# print(x)
# print(y)

x = np.array(x)
y = np.array(y)
x_arr = np.column_stack((np.ones(x.shape[0]), x))

X_train, X_test, y_train, y_test = train_test_split(x_arr, y, shuffle=False)


model = xgb.XGBRegressor()
model.fit(X_train, y_train)

predictions = model.predict(X_test)

beta_3_rmse = rmse(y_test, predictions)

beta_rmses = [beta_1_rmse, beta_2_rmse, beta_3_rmse]
for beta, beta_rmse in enumerate(beta_rmses, start=1):
    print(f"  Beta: {beta}")
    print(f"  Forecast 1 day, RMSE: {beta_rmse}")



  Beta: 1
  Forecast 1 day, RMSE: 1.5545824888555235
  Beta: 2
  Forecast 1 day, RMSE: 1.0434699597023345
  Beta: 3
  Forecast 1 day, RMSE: 3.2982544231157336
